# First-Pass Batch 2

This notebook handles the deterministic first pass for batch 2 of 4. It does not retry failed rows.


## Drive And Repo Setup

Mount Drive, clone the repo into the runtime, and copy the required CSV inputs into the checkout.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output root
- Rerun-safe: Yes. It reclones the repo and recopies the inputs cleanly.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
OUTPUT_ROOT = DRIVE_ROOT / "svg-kaggle-comptetition" / "submission_outputs"

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    if destination.exists():
        continue

    preferred_sources = [
        DRIVE_ROOT / "svg-kaggle-comptetition" / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    copied = False
    for candidate in preferred_sources:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            copied = True
            break
    if not copied:
        for candidate in DRIVE_ROOT.rglob(required_name):
            if candidate.is_file():
                shutil.copy2(candidate, destination)
                copied = True
                break
    if not copied:
        raise FileNotFoundError(
            f"Could not find {required_name} in Google Drive. Copy it into {CHECKOUT_PATH} manually and rerun this cell."
        )

os.environ["SVG_KAGGLE_REPO_ROOT"] = str(CHECKOUT_PATH)
sys.path.insert(0, str(CHECKOUT_PATH))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Drive output root: {OUTPUT_ROOT}")


## Package Check

Install any missing runtime packages required by the shared helper module and this notebook.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs missing packages.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "cairosvg": "cairosvg",
    "skimage": "scikit-image",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Load Helpers, Recommendation, And Batch Slice

Import the shared helper module, load the analysis recommendation, and select this notebook's fixed quarter of `test.csv`.

- Reads: Repo checkout, helper module, analysis recommendation, competition CSV files
- Writes: In-memory batch slice only
- Rerun-safe: Yes. It is deterministic because the batch boundaries are fixed.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import SVG, Markdown, display

import submission_pipeline as pipeline

pipeline.set_global_seed(1337)
PATHS = pipeline.resolve_pipeline_paths(Path(os.environ["SVG_KAGGLE_REPO_ROOT"]), OUTPUT_ROOT)
CONFIG = pipeline.GenerationConfig(verbose_progress=True)
TEST_DF, SAMPLE_SUBMISSION_DF, TRAIN_DF = pipeline.load_competition_frames(PATHS)

print(f"Repo root: {PATHS.repo_root}")
print(f"Output root: {PATHS.output_root}")

BATCH_ID = 2
ANALYSIS_RECOMMENDATION = pipeline.load_analysis_recommendation(PATHS)
GENERATION_MODE = ANALYSIS_RECOMMENDATION.get("recommended_generation_mode", "body_only")
BATCH_DF = pipeline.select_batch_df(TEST_DF, batch_id=BATCH_ID, batch_count=4)
BATCH_DIR = pipeline.batch_output_dir(PATHS, BATCH_ID)
print(f"Recommended generation mode: {GENERATION_MODE}")
print(f"Batch rows: {len(BATCH_DF)}")
display(BATCH_DF[["id", "prompt", "row_index"]].head())


## Load Model

Load the base model plus the LoRA adapter into the current Colab GPU runtime for first-pass generation.

- Reads: Base model on Hugging Face, LoRA adapter from repo
- Writes: In-memory model state only
- Rerun-safe: No. It is safe to rerun, but it reloads the model and costs time.


In [ ]:
RUNTIME = pipeline.load_lora_runtime(PATHS.adapter_dir)
print(f"Loaded runtime on: {RUNTIME.runtime_device}")


## Deterministic First Pass

Run one deterministic generation attempt per row in this batch and save the success/failure artifacts for the merge notebook.

- Reads: Selected batch rows, model runtime
- Writes: Batch output CSV artifacts under this batch directory
- Rerun-safe: Yes. It recomputes and overwrites this batch's artifacts.


In [ ]:
FIRST_PASS_RESULTS_DF = pipeline.run_first_pass_batch(
    BATCH_DF,
    RUNTIME,
    batch_id=BATCH_ID,
    generation_mode=GENERATION_MODE,
    config=CONFIG,
)
pipeline.save_batch_outputs(FIRST_PASS_RESULTS_DF, BATCH_DIR)
print(f"Wrote batch artifacts to: {BATCH_DIR}")


## Batch Summary

Show the first-pass success/failure totals and inspect a few failed rows with their gate errors.

- Reads: Batch result artifacts in memory
- Writes: Nothing
- Rerun-safe: Yes. Read-only display cell.


In [ ]:
display(pipeline.summarize_batch_results(FIRST_PASS_RESULTS_DF))
FAILED_DF = FIRST_PASS_RESULTS_DF[~FIRST_PASS_RESULTS_DF["first_pass_success"]].copy()
if FAILED_DF.empty:
    print("No failed rows in this batch.")
else:
    display(FAILED_DF[["id", "row_index", "generation_mode", "gate_error"]].head(20))
